# Stage 02 — Data Validation
Prototype notebook for the data validation component.
Run after `1_data_ingestion.ipynb` is complete.

In [32]:
import os
#os.chdir("../")
print("Working directory:", os.getcwd())

Working directory: /home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk


In [33]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [34]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories, logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS
        create_directories([config.root_dir])
        return DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema,
        )

In [35]:
import os
import pandas as pd
from src.creditrisk.utils import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True

            data_file = os.path.join(self.config.unzip_data_dir, "train.csv")
            df = pd.read_csv(data_file)  # removed skiprows=1

            # Drop identifier columns not in schema
            df = df.drop(columns=["customer_id", "name"], errors="ignore")

            all_schema_columns = list(self.config.all_schema.keys())
            df_columns = df.columns.tolist()

            for col in df_columns:
                if col not in all_schema_columns:
                    validation_status = False
                    logger.info(f"Column '{col}' not found in schema")

            if validation_status:
                logger.info("All columns validated successfully against schema")

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation Status: {'Passed' if validation_status else 'Failed'}")

            return validation_status
        except Exception as e:
            logger.exception(e)
            raise e

In [36]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-05-04 19:36:51,406: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-04 19:36:51,412: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-04 19:36:51,418: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-04 19:36:51,421: INFO: common]: created directory at: artifacts
[2026-05-04 19:36:51,423: INFO: common]: created directory at: artifacts/data_validation
[2026-05-04 19:36:51,570: INFO: 141225029]: All columns validated successfully against schema


In [37]:
with open("artifacts/data_validation/status.txt") as f:
    print(f.read())

Validation Status: Passed
